# Analyze your own model with the local/global geometry framework

This notebook shows how to apply the paper's analyses to **any model you
have** — your own recurrent network, a new architecture, a different layer.
No knowledge of the rest of this repo is required.

**The entire contract is two tensors and a label vector:**

| What | Shape | Meaning |
|---|---|---|
| `X_baseline` | `[N, D]` | activations for N images at the *earlier* stage (first pass / early timestep) |
| `X_after` | `[N, D]` | activations for the **same N images, in the same order**, at the *later* stage |
| `labels` | `[N]` | integer category label per image |

That's it. Rows must correspond to the same images in the same order across
the two tensors — that is the one thing the analysis cannot check for you.
(Feedforward models with no time dimension just have a single `X` and can
join the between-prototype comparisons, but not the baseline→after dynamics.)

We'll walk through three steps:
1. run the geometry metrics on a single model (demo data so it runs anywhere),
2. extract real activations from a PyTorch model with a forward hook,
3. add your model *alongside the paper's models* in the comparison figures.

## Step 1 — the metrics, on demo data

So this notebook runs without any downloads, we first fabricate a "model"
whose recurrence compacts categories locally (exemplars drawn toward their
prototype) while leaving global structure intact — i.e. LRM/LRA-like
dynamics. Swap in your real tensors and nothing else changes.

In [ ]:
try:
    import repgeo
except ImportError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve().parent))
    import repgeo

import torch

# ---- demo data: 100 categories x 50 images, 512-d features ----
torch.manual_seed(0)
n_classes, per_class, dim = 100, 50, 512
labels = torch.arange(n_classes).repeat_interleave(per_class)        # [5000]
prototypes = torch.randn(n_classes, dim)                             # one center per category
X_baseline = prototypes[labels] + 1.2 * torch.randn(len(labels), dim)
# "recurrence": pull every exemplar 30% of the way toward its prototype
X_after = 0.7 * X_baseline + 0.3 * prototypes[labels]

print(X_baseline.shape, X_after.shape, labels.shape)

`pair_geometry_metrics` computes the paper's five Table-2 numbers for one
baseline→after pair. For this demo we expect a negative Δ Cluster Size
(compaction) with global structure preserved (Proto RDM ρ near 1).

In [ ]:
from repgeo import pair_geometry_metrics

metrics = pair_geometry_metrics(X_baseline, X_after, labels)
for name, value in metrics.items():
    print(f"  {name:22s} {value:+.4f}")

And the corresponding Figure-1-style panel — local cluster-size-change KDE
plus the prototype-shift PCA — comes from the same function the paper used
for its supplement figure:

In [ ]:
from repgeo import plotting as P

P.plot_convrnn_supplement(   # generic baseline/after panel, despite the name
    X_baseline, X_after, labels,
    before_label="Baseline", after_label="After recurrence")

The significance tests work on raw tensors too:

In [ ]:
from repgeo.stats import local_cluster_lme, between_separation_permutation
from repgeo import compute_prototypes, exemplar_proto_dists

classes = torch.unique(labels, sorted=True).tolist()
d_b = exemplar_proto_dists(X_baseline, compute_prototypes(X_baseline, labels), labels)
d_a = exemplar_proto_dists(X_after,    compute_prototypes(X_after, labels),    labels)

print("Local LME:", local_cluster_lme(d_b, d_a, labels))

perm = between_separation_permutation(X_baseline, X_after, labels, n_perm=500)
print(f"Global permutation: obs={perm['obs_pct']:+.2f}%  p(two-tailed)={perm['p_two_tailed']:.4f}")

## Step 2 — getting real activations out of a PyTorch model

If you don't already have activation tensors, the standard recipe is a
**forward hook**: a small function PyTorch calls whenever a chosen layer
runs, letting you copy out its activations. Below is the minimal pattern —
this is exactly what `scripts/extraction/extract_torch_models.py` does at
scale, with file saving and a proper dataloader.

```python
import torch

model = ...                       # your model, .eval() mode
layer = model.classifier[5]       # the layer you want activations from

storage = []
hook = layer.register_forward_hook(
    lambda module, inp, out: storage.append(out.detach().cpu()))

with torch.inference_mode():
    for imgs, _ in dataloader:               # shuffle=False !
        model(imgs)                          # hook fills `storage`
hook.remove()

X = torch.cat(storage)                       # [N, D]
```

Two things matter for the geometry analysis to be valid:

* **`shuffle=False`** in the dataloader, so the baseline and after tensors
  are row-aligned with each other and with `labels`.
* For a recurrent model, run the loop once per stage (e.g. pass 1 and
  pass 3) — or, if your model exposes per-timestep states, collect the two
  timesteps in a single loop.

To reproduce the paper's exact stimulus set (100 ImageNet-val categories ×
50 images), see `scripts/build_stimulus_set.py` — you need your own
ImageNet copy. The geometry framework itself works with any image set that
has multiple exemplars per category.

## Step 3 — your model next to the paper's models

The comparison figures all read from one dictionary. Adding your model is
one entry (recurrent models have `"baseline"` and `"after"`; feedforward
models just `"baseline"`):

In [ ]:
features = {
    "MyModel": {
        "logits": {},   # fill in if you have 1000-way logits
        "penult": {"baseline": X_baseline, "after": X_after},
    },
}

# If you downloaded the paper's activations, load them too and your model
# appears in the same tables and figures (uncomment):
#
# paper_features, paper_labels = repgeo.load_features("../activations")
# features.update(paper_features)

from repgeo import geometry_summary_table, config

# The summary analyzes logits by default; tell it MyModel lives at the
# penultimate layer (the paper does the same for CORnet-RT).
config.REP_OVERRIDES["MyModel"] = "penult"

geometry_summary_table(features, labels, models=["MyModel"])

For your model to get its own color and pretty name in the ridge/MDS
figures, register it once (otherwise it's drawn in gray with its dict key):

In [ ]:
from repgeo import config, compute_cluster_sizes_and_rdms

config.MODEL_ORDER.insert(0, "MyModel")
config.MODEL_COLOR_MAP["MyModel"] = {"baseline": "#7A4FA3", "after": "#B07FE0"}
config.DISPLAY_NAMES["MyModel baseline"] = "MyModel (baseline)"
config.DISPLAY_NAMES["MyModel after"] = "MyModel (after)"

_, between = compute_cluster_sizes_and_rdms(features, labels, rep="penult")
P.plot_separation_ridge(between, xlabel="Between-prototype cosine distance")

That's the whole loop: **two row-aligned `[N, D]` tensors + labels →
Table-2 metrics, significance tests, and every comparison figure.**

If you use this framework, please cite the paper (see README). Questions and
issues are welcome on the GitHub tracker.